<a href="https://colab.research.google.com/github/nashranoor98/hospital-readmission-prediction/blob/main/CaseStudy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hospital Readmission Prediction
Predicting 30-day hospital readmission using Logistic Regression with L2 regularization.


## 1. Importing and Studying Data


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [
    "data/diabetic_data.csv",
    "/content/data/diabetic_data.csv",
    "/content/hospital_data/diabetic_data.csv",
    "diabetic_data.csv"
]
DATA_URL = "https://raw.githubusercontent.com/moggirain/Hospital_readmission_prediction/master/diabetic_data.csv"
for path in DATA_PATHS:
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    DATA_PATH = DATA_URL

data = pd.read_csv(DATA_PATH).replace("?", np.nan)
print("Dataset shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())


In [ ]:
print(data.info())
print("\nDescriptive statistics:")
print(data.describe(include="all").T.head(15))
print("\nMissing values (top 10):")
print(data.isnull().sum().sort_values(ascending=False).head(10))
print("\nDuplicate rows:", data.duplicated().sum())


## 2. EDA and Visualisation
The following graphs are generated from the hospital dataset used for this case study.


### Readmission Distribution

![Readmission Distribution](graphs/01_readmission_distribution.svg)


### Age Group Distribution

![Age Group Distribution](graphs/02_age_group_distribution.svg)


### Numeric Feature Distributions

![Numeric Feature Distributions](graphs/03_numeric_distributions.svg)


### Gender Distribution

![Gender Distribution](graphs/04_gender_distribution.svg)


### Correlation Heatmap

![Correlation Heatmap](graphs/05_correlation_heatmap.svg)


In [ ]:
print(data["readmitted"].value_counts())

numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient"
]


## 3. Splitting and Scaling


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data["target_30day"] = (data["readmitted"] == "<30").astype(int)
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "target_30day"]
X = data.drop(columns=drop_cols)
y = data["target_30day"]
high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Features retained:", X.shape[1])


## 4. Training and Evaluating Model


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, solver="liblinear"))
])
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


### ROC Curve

![ROC Curve](graphs/06_roc_curve.svg)


### Confusion Matrix

![Confusion Matrix](graphs/07_confusion_matrix.svg)


## 5. False Negative vs False Positive
False negatives can be clinically costly because a high-risk patient may be missed. False positives may lead to additional follow-up or resource use. Therefore, threshold selection should consider the clinical cost of missed readmissions.

### Verified executed results
- Dataset: **101,766 rows × 50 columns**
- 30-day target: **11,357 positive / 90,409 negative**
- Train/test split: **81,412 / 20,354**
- ROC-AUC: **0.6462**
- Confusion matrix: **[[18039, 44], [2229, 42]]**
